# **Project Name** - Uber Request Data Analysis (EDA)


##### **Project Type** - EDA (Exploratory Data Analysis)
##### **Contribution** - Individual
##### **Team Member 1 -** Pardeep | BCA (AI & ML) | Jaipur National University

# **Project Summary -**

This project performs a complete Exploratory Data Analysis (EDA) on Uber ride request data from 2016. The dataset captures real-world ride requests made by customers, including details like pickup location (Airport or City), driver assignment, trip status (Completed, Cancelled, No Cars Available), and timestamps for both request and drop-off.

The primary motivation behind this analysis is to understand the demand-supply dynamics of Uber's ride-hailing service. A major operational challenge for Uber is ensuring that drivers are available where and when customers need them. This project investigates when and where demand peaks, where supply consistently falls short, and what operational patterns lead to trip failures.

The analysis follows the UBM framework — Univariate, Bivariate, and Multivariate Analysis — to progressively uncover insights:

- **Univariate Analysis** examines individual variables such as trip status distribution, pickup point frequency, hourly request volume, and trip duration distribution.
- **Bivariate Analysis** explores relationships between pairs of variables — for example, how trip status differs between Airport and City pickups, or how completion rates change by hour of day.
- **Multivariate Analysis** combines three or more dimensions, such as comparing pickup point and time slot together against trip outcomes using heatmaps.

Key feature engineering was performed including extraction of Hour, Day, and creation of Time Slots (Morning Rush, Late Morning, Afternoon, Evening Rush, Late Night) and calculation of Trip Duration in minutes.

Major findings include: a severe Evening Rush supply shortage particularly at the Airport, high driver cancellation rates in the City during Morning Rush, and midday hours showing the best trip completion rates. The analysis concludes with actionable business recommendations including surge pricing during peak hours, driver incentive restructuring, and targeted airport deployment strategies.

Tools used: Python (Pandas, NumPy, Matplotlib, Seaborn), SQL (SQLite), and Excel for dashboard reporting.

# **GitHub Link -**

https://github.com/pardeep0011/Data-Analyst-

# **Problem Statement**


Uber frequently faces situations where customer ride requests cannot be fulfilled due to driver cancellations or a complete absence of available cars. This results in poor customer experience, lost revenue, and operational inefficiency.

The core problem is identifying **when**, **where**, and **why** ride requests fail — and using that information to recommend strategies that reduce unfulfilled requests, balance driver supply with customer demand, and improve overall service reliability across different pickup locations and time periods.

#### **Define Your Business Objective?**

**Primary Objective:** To analyze Uber's 2016 ride request data and identify demand-supply gaps, trip failure patterns, and operational inefficiencies across different locations and time periods.

**Secondary Objectives:**
1. Identify peak demand hours and time slots to support smarter driver allocation.
2. Compare Airport vs. City pickup performance and uncover location-specific issues.
3. Measure hourly trip completion rates to pinpoint critical failure windows.
4. Detect patterns in driver cancellations and 'No Cars Available' scenarios.
5. Provide data-driven recommendations to increase trip completion rate and revenue.
6. Build a reusable Python EDA pipeline that can be extended to future datasets.

# **General Guidelines** : -  

1. Well-structured, formatted, and commented code is required.
2. Exception Handling, Production Grade Code & Deployment Ready Code will be a plus.
3. Each and every logic should have proper comments.
4. For each chart answer: Why picked, Insights found, Business Impact.
5. Minimum 20 logical & meaningful charts required.
6. Follow UBM Rule: Univariate → Bivariate → Multivariate Analysis.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# ─── Import Libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# Set global plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')

### Dataset Loading

In [ ]:
# ─── Load Dataset ────────────────────────────────────────────────────────────
try:
    df = pd.read_csv('Uber Request Data.csv')
    print(f'Dataset loaded successfully! Shape: {df.shape}')
except FileNotFoundError as e:
    print(f'Error loading file: {e}')
    raise

### Dataset First View

In [ ]:
# ─── Dataset First Look ──────────────────────────────────────────────────────
df.head(10)

### Dataset Rows & Columns count

In [ ]:
# ─── Dataset Shape ───────────────────────────────────────────────────────────
print(f'Total Rows    : {df.shape[0]}')
print(f'Total Columns : {df.shape[1]}')

### Dataset Information

In [ ]:
# ─── Dataset Info ────────────────────────────────────────────────────────────
df.info()

### Dataset Summary

In [ ]:
# ─── Statistical Summary ─────────────────────────────────────────────────────
df.describe(include='all')

### Duplicate Values

In [ ]:
# ─── Check for Duplicate Rows ────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f'Total Duplicate Rows: {duplicates}')

### Missing Values / Null Values

In [ ]:
# ─── Missing Value Analysis ──────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_df[missing_df['Missing Count'] > 0])

### What did you know about your dataset?

- The dataset contains **6,745 Uber ride request records** with 6 columns: Request id, Pickup point, Driver id, Status, Request timestamp, Drop timestamp.
- **Status** has 3 categories: Trip Completed, Cancelled, No Cars Available.
- **Pickup point** has 2 values: Airport and City.
- **Driver id** and **Drop timestamp** have missing values — this is expected since cancelled and unavailable rides have no driver or drop time.
- No duplicate records were found in the dataset.
- Timestamps need to be parsed and feature-engineered to extract Hour, Day, and Time Slot for meaningful analysis.

## ***2. Understanding Your Variables***

In [ ]:
# ─── Column Names and Data Types ─────────────────────────────────────────────
print(df.dtypes)

In [ ]:
# ─── Unique Values in Categorical Columns ────────────────────────────────────
print('Pickup Point values :', df['Pickup point'].unique())
print('Status values       :', df['Status'].unique())

### Variables Description

| Variable | Type | Description |
|---|---|---|
| Request id | Numerical | Unique identifier for each ride request |
| Pickup point | Categorical | Location of ride request — Airport or City |
| Driver id | Numerical | ID of assigned driver (missing if no driver assigned) |
| Status | Categorical | Outcome — Trip Completed, Cancelled, No Cars Available |
| Request timestamp | DateTime | Date and time when the ride was requested |
| Drop timestamp | DateTime | Date and time when the trip ended (missing for non-completed trips) |

### Why is the analysis required?

A significant portion of Uber ride requests are never fulfilled — either due to driver cancellations or no available cars. Without understanding when and where these failures occur, Uber cannot make targeted operational improvements. This EDA is required to:

1. Quantify the scale of demand-supply mismatch.
2. Pinpoint the exact hours and locations where failures concentrate.
3. Provide evidence-based recommendations to reduce ride failures and increase revenue.

## ***3. Data Wrangling***

In [ ]:
# ─── Parse Timestamps ────────────────────────────────────────────────────────
# Handle mixed date formats in the timestamp columns
df['Request timestamp'] = pd.to_datetime(df['Request timestamp'], dayfirst=True, errors='coerce')
df['Drop timestamp']    = pd.to_datetime(df['Drop timestamp'],    dayfirst=True, errors='coerce')
print('Timestamps parsed successfully.')

In [ ]:
# ─── Feature Engineering ─────────────────────────────────────────────────────

# Extract hour from request timestamp (used for hourly demand analysis)
df['Hour'] = df['Request timestamp'].dt.hour

# Extract date from request timestamp
df['Date'] = df['Request timestamp'].dt.date

# Extract weekday name
df['Day'] = df['Request timestamp'].dt.day_name()

# Calculate trip duration in minutes (only valid for completed trips)
df['Trip Duration (mins)'] = (
    (df['Drop timestamp'] - df['Request timestamp']).dt.total_seconds() / 60
).round(2)

# Create time slots for business-friendly categorization
def assign_time_slot(hour):
    if 4 <= hour < 9:
        return 'Morning Rush'
    elif 9 <= hour < 12:
        return 'Late Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 22:
        return 'Evening Rush'
    else:
        return 'Late Night'

df['Time Slot'] = df['Hour'].apply(assign_time_slot)

print('Feature engineering complete!')
print(df[['Hour','Day','Trip Duration (mins)','Time Slot']].head())

In [ ]:
# ─── Handle Missing Values ───────────────────────────────────────────────────
# Driver id is missing for 'No Cars Available' — this is structurally correct, not an error
# Drop timestamp is missing for Cancelled and No Cars Available — also structurally correct
# We retain these rows as-is; dropping them would lose valuable failure data

print('Missing values after feature engineering:')
print(df.isnull().sum())

### What all manipulations have you done and why?

1. **Timestamp Parsing** — Converted `Request timestamp` and `Drop timestamp` from string to datetime using `dayfirst=True` to handle the mixed DD/MM/YYYY HH:MM format correctly.
2. **Hour Extraction** — Extracted hour of request to enable hourly demand and completion rate analysis.
3. **Day Extraction** — Extracted weekday name to check for day-of-week demand patterns.
4. **Trip Duration Calculation** — Computed duration in minutes as `(Drop - Request) / 60`. This is null for non-completed trips, which is expected.
5. **Time Slot Creation** — Grouped hours into 5 business-meaningful time bands (Morning Rush, Late Morning, Afternoon, Evening Rush, Late Night) to make demand patterns more interpretable.
6. **Missing Values Retained** — Missing Driver id and Drop timestamp are structurally valid for failed trips and were not imputed or dropped.

## ***4. Data Vizualization, Storytelling & Experimenting with charts***

### 4.1 Univariate Analysis

#### Chart - 1 : Trip Status Distribution (Count Plot)

In [ ]:
# ─── Chart 1: Status Distribution ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
status_counts = df['Status'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#e67e22']
bars = ax.bar(status_counts.index, status_counts.values, color=colors, edgecolor='black', linewidth=0.8)

# Annotate bars with count
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Overall Trip Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Trip Status')
ax.set_ylabel('Number of Requests')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A bar chart is ideal for comparing discrete categories. It gives an immediate visual sense of the proportional split between completed, cancelled, and unavailable rides.

##### 2. What is/are the insight(s) found from the chart?
- 'No Cars Available' and 'Cancelled' together account for more than 58% of all requests — meaning over half of all customer ride requests are unfulfilled.
- 'Trip Completed' is the single largest category (~42%) but is far from dominant, indicating a serious service gap.

##### 3. Will the gained insights help creating a positive business impact?
Yes — identifying that 58%+ requests fail is a critical business finding. Reducing the failure rate by even 10% would significantly increase revenue and customer retention. This chart establishes the core problem that drives all subsequent analysis.

#### Chart - 2 : Hourly Request Volume (Line Chart)

In [ ]:
# ─── Chart 2: Requests by Hour ───────────────────────────────────────────────
hourly = df.groupby('Hour').size().reset_index(name='Requests')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly['Hour'], hourly['Requests'], marker='o', color='#3498db', linewidth=2.5, markersize=7)
ax.fill_between(hourly['Hour'], hourly['Requests'], alpha=0.15, color='#3498db')
ax.set_title('Total Ride Requests by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Requests')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A line chart with area fill is best suited to show temporal trends over a continuous variable (hour). It clearly shows peaks and troughs across the 24-hour cycle.

##### 2. What is/are the insight(s) found from the chart?
- Two distinct demand peaks exist: **5 AM – 9 AM** (Morning Rush) and **5 PM – 9 PM** (Evening Rush).
- Demand is very low between **11 PM – 4 AM**.
- The evening peak is slightly higher than the morning peak.

##### 3. Business Impact?
Uber should ensure maximum driver availability during these two peak windows. Incentive bonuses for being online during 5–9 AM and 5–9 PM could reduce supply shortages precisely when demand is highest.

#### Chart - 3 : Pickup Point Distribution (Pie Chart)

In [ ]:
# ─── Chart 3: Pickup Point Distribution ─────────────────────────────────────
pickup_counts = df['Pickup point'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(pickup_counts, labels=pickup_counts.index, autopct='%1.1f%%',
       colors=['#9b59b6', '#1abc9c'], startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Ride Requests by Pickup Point', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A pie chart is appropriate here since we have only two mutually exclusive categories (Airport vs City) and want to show their proportional share of total requests.

##### 2. Insights?
City pickup requests (~53%) slightly exceed Airport requests (~47%). Both locations contribute significantly — neither can be deprioritized.

##### 3. Business Impact?
Uber needs a balanced operational strategy for both locations. The near-equal split means Airport-specific and City-specific problems both have substantial customer impact.

#### Chart - 4 : Trip Duration Distribution (Histogram)

In [ ]:
# ─── Chart 4: Trip Duration Histogram ───────────────────────────────────────
completed = df[df['Status'] == 'Trip Completed']['Trip Duration (mins)'].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(completed, bins=30, color='#27ae60', edgecolor='black', alpha=0.8)
ax.axvline(completed.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {completed.mean():.1f} mins')
ax.axvline(completed.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {completed.median():.1f} mins')
ax.set_title('Trip Duration Distribution (Completed Trips)', fontsize=14, fontweight='bold')
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A histogram with mean/median lines is the standard way to understand distribution shape, central tendency, and spread of a continuous numerical variable like trip duration.

##### 2. Insights?
- Average trip duration is approximately 52–55 minutes.
- Distribution is fairly symmetric with slight right skew.
- Most trips fall between 20 and 80 minutes.

##### 3. Business Impact?
Uber can use average trip duration for driver earnings estimation and ETA prediction, helping build better driver incentive models and customer communication features.

#### Chart - 5 : Requests by Day of Week (Bar Chart)

In [ ]:
# ─── Chart 5: Requests by Day ────────────────────────────────────────────────
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']
day_counts = df['Day'].value_counts().reindex(day_order)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(day_counts.index, day_counts.values, color='#e67e22', edgecolor='black')
ax.set_title('Ride Requests by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Number of Requests')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Bar chart is ideal for comparing demand volume across discrete categories (days of the week).

##### 2. Insights?
Demand is relatively stable across all weekdays with no single day showing a dramatic spike. This suggests Uber's operational planning does not need day-of-week adjustments as much as hour-of-day adjustments.

##### 3. Business Impact?
Positive — daily demand consistency means driver scheduling can be standardized across weekdays. Resources should be focused on time-of-day optimization rather than day-of-week optimization.

### 4.2 Bivariate Analysis

#### Chart - 6 : Status by Pickup Point (Grouped Bar Chart)

In [ ]:
# ─── Chart 6: Status by Pickup Point ────────────────────────────────────────
status_pickup = df.groupby(['Pickup point','Status']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
status_pickup.plot(kind='bar', ax=ax, color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='black')
ax.set_title('Trip Status by Pickup Point', fontsize=14, fontweight='bold')
ax.set_xlabel('Pickup Point')
ax.set_ylabel('Number of Requests')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Status')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A grouped bar chart allows direct comparison of trip status counts across two pickup points simultaneously — perfect for categorical vs categorical comparison.

##### 2. Insights?
- **Airport**: Dominated by 'No Cars Available' — drivers are simply not present when passengers need rides after flights.
- **City**: Dominated by 'Cancelled' — drivers accept but then cancel City rides, likely preferring longer airport trips.

##### 3. Business Impact?
This is one of the most impactful findings. It reveals two distinct problems requiring two different solutions: increase airport driver supply vs. reduce city cancellation incentives. Addressing both could dramatically improve completion rates.

#### Chart - 7 : No Cars Available by Hour (Bar Chart)

In [ ]:
# ─── Chart 7: No Cars Available by Hour ─────────────────────────────────────
no_cars = df[df['Status'] == 'No Cars Available'].groupby('Hour').size()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(no_cars.index, no_cars.values, color='#c0392b', edgecolor='black')
ax.set_title('"No Cars Available" Requests by Hour', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Count')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Bar chart over hours clearly shows which specific hours suffer from supply shortages — enabling targeted operational response.

##### 2. Insights?
Supply shortages peak between **5 PM and 9 PM** — these 4 hours generate the highest number of 'No Cars Available' responses in the entire dataset.

##### 3. Business Impact?
Surge pricing and driver incentive bonuses specifically during 5–9 PM could pull more drivers online exactly when shortage is worst, directly converting lost revenue into completed trips.

#### Chart - 8 : Trip Completion Rate by Hour (Line Chart)

In [ ]:
# ─── Chart 8: Completion Rate by Hour ───────────────────────────────────────
hourly_total     = df.groupby('Hour').size()
hourly_completed = df[df['Status'] == 'Trip Completed'].groupby('Hour').size()
completion_rate  = (hourly_completed / hourly_total * 100).fillna(0).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(completion_rate.index, completion_rate.values, marker='s', color='#16a085', linewidth=2.5, markersize=7)
ax.axhline(completion_rate.mean(), color='red', linestyle='--', label=f'Avg: {completion_rate.mean():.1f}%')
ax.set_title('Trip Completion Rate by Hour (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Completion Rate (%)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Line chart with average reference line shows not just the trend but also which hours fall below the average completion rate — making underperforming hours immediately identifiable.

##### 2. Insights?
- Completion rate is highest during **10 AM – 4 PM** (midday hours).
- Lowest completion rates occur during **5 PM – 9 PM** and early morning (1–4 AM).
- Evening Rush is both the highest demand AND the lowest completion window — worst-case scenario.

##### 3. Business Impact?
Targeting the 5–9 PM window for both supply-side (driver incentives) and demand-side (advance booking prompts, surge pricing) interventions would have maximum positive impact.

#### Chart - 9 : Cancellations by Hour (Bar Chart)

In [ ]:
# ─── Chart 9: Cancellations by Hour ─────────────────────────────────────────
cancelled_hourly = df[df['Status'] == 'Cancelled'].groupby('Hour').size()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(cancelled_hourly.index, cancelled_hourly.values, color='#e67e22', edgecolor='black')
ax.set_title('Driver Cancellations by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Cancellations')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Bar chart over hours makes it easy to see which hours suffer most from driver cancellations — a supply behavior pattern distinct from 'No Cars Available'.

##### 2. Insights?
Cancellations peak during **5 AM – 9 AM** (Morning Rush). This is mostly City-based cancellations where drivers prefer to wait for airport pickups instead of short city trips.

##### 3. Business Impact?
Introducing cancellation penalties during Morning Rush hours and city-specific trip bonuses could reduce this pattern. Negative growth risk: if penalties are too harsh, drivers may go offline instead of cancelling — reducing overall supply.

#### Chart - 10 : Trip Duration by Pickup Point (Box Plot)

In [ ]:
# ─── Chart 10: Trip Duration by Pickup Point ─────────────────────────────────
completed_df = df[df['Status'] == 'Trip Completed']

fig, ax = plt.subplots(figsize=(8, 6))
completed_df.boxplot(column='Trip Duration (mins)', by='Pickup point', ax=ax,
                     patch_artist=True,
                     boxprops=dict(facecolor='#3498db', color='black'),
                     medianprops=dict(color='red', linewidth=2))
ax.set_title('Trip Duration by Pickup Point', fontsize=14, fontweight='bold')
ax.set_xlabel('Pickup Point')
ax.set_ylabel('Duration (minutes)')
plt.suptitle('')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Box plot shows median, quartile range, and outliers simultaneously — ideal for comparing the distribution of a numerical variable across two groups.

##### 2. Insights?
Airport trips tend to be longer in duration than City trips. This explains driver preference for airport pickups — longer trips mean higher earnings per ride.

##### 3. Business Impact?
To make City trips equally attractive, Uber could introduce per-minute pricing adjustments or guaranteed minimum earnings for short trips to reduce driver bias toward airport rides.

#### Chart - 11 : Status Distribution by Time Slot (Stacked Bar)

In [ ]:
# ─── Chart 11: Status Across Time Slots (Stacked) ───────────────────────────
slot_order = ['Morning Rush','Late Morning','Afternoon','Evening Rush','Late Night']
slot_status = df.groupby(['Time Slot','Status']).size().unstack(fill_value=0)
slot_status = slot_status.reindex(slot_order)

fig, ax = plt.subplots(figsize=(12, 6))
slot_status.plot(kind='bar', stacked=True, ax=ax,
                 color=['#e74c3c','#f1c40f','#2ecc71'], edgecolor='black')
ax.set_title('Trip Status Distribution Across Time Slots', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Slot')
ax.set_ylabel('Number of Requests')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20)
ax.legend(title='Status', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Stacked bar chart lets us see both the total request volume AND the status composition within each time slot in a single view.

##### 2. Insights?
- Evening Rush has the highest total volume and the largest proportion of 'No Cars Available'.
- Morning Rush has the highest cancellations relative to completions.
- Afternoon has the best completion-to-failure ratio.

##### 3. Business Impact?
Time-slot-specific driver incentives would be more effective than flat incentives. Afternoon bonuses are less needed; Evening Rush and Morning Rush need targeted intervention.

#### Chart - 12 : Trip Duration Outlier Detection (Box Plot)

In [ ]:
# ─── Chart 12: Trip Duration Outliers ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(completed['Trip Duration (mins)'] if isinstance(completed, pd.DataFrame)
           else completed,
           vert=False, patch_artist=True,
           boxprops=dict(facecolor='#9b59b6'),
           medianprops=dict(color='white', linewidth=2))
ax.set_title('Trip Duration Outlier Detection', fontsize=14, fontweight='bold')
ax.set_xlabel('Duration (minutes)')
plt.tight_layout()
plt.show()

# Quantify outliers using IQR method
Q1, Q3 = completed.quantile(0.25), completed.quantile(0.75)
IQR = Q3 - Q1
outliers = completed[(completed < Q1 - 1.5*IQR) | (completed > Q3 + 1.5*IQR)]
print(f'Number of outlier trips: {len(outliers)}')
print(f'Outlier trip durations (mins):\n{outliers.describe()}')

##### 1. Why did you pick the specific chart?
Box plot is the standard chart for detecting outliers — data points beyond 1.5×IQR from the quartile boundaries are visually shown as individual dots.

##### 2. Insights?
Very few extreme outliers exist. The dataset is largely clean with no suspicious trip durations that would indicate data quality issues.

##### 3. Business Impact?
Clean data means analysis results are reliable. The few outliers (very long trips) could indicate detours or data logging errors and can be flagged for quality review.

#### Chart - 13 : Missing Values Heatmap

In [ ]:
# ─── Chart 13: Missing Values Heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False, ax=ax)
ax.set_title('Missing Values Heatmap (Yellow = Missing)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A heatmap of nulls provides an instant visual of which columns have missing data and whether missingness follows a pattern.

##### 2. Insights?
- `Driver id` and `Drop timestamp` both have structured missingness — they are always missing together for the same rows (cancelled and no-car rides).
- This is structurally valid missingness (Missing Not At Random), not random data loss.

##### 3. Business Impact?
No data imputation required. The missingness pattern itself is informative — it confirms which requests were definitively unfulfilled.

#### Chart - 14 : Correlation Heatmap

In [ ]:
# ─── Chart 14: Correlation Heatmap ──────────────────────────────────────────
# Encode categorical columns for correlation
df_encoded = df.copy()
df_encoded['Status_encoded']       = df['Status'].map({'Trip Completed':2,'Cancelled':1,'No Cars Available':0})
df_encoded['Pickup_encoded']       = df['Pickup point'].map({'City':0,'Airport':1})

corr_cols = ['Hour','Pickup_encoded','Status_encoded','Trip Duration (mins)']
corr_matrix = df_encoded[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A correlation heatmap shows the strength and direction of linear relationships between numerical/encoded variables — standard for understanding feature interdependencies in EDA.

##### 2. Insights?
- Hour has a moderate negative correlation with trip completion status, confirming that later hours have worse completion rates.
- Pickup point has some correlation with trip status, supporting the Airport vs. City findings from earlier charts.

### 4.3 Multivariate Analysis

#### Chart - 15 : Pickup Point × Time Slot Heatmap (Request Volume)

In [ ]:
# ─── Chart 15: Pickup Point vs Time Slot Heatmap ─────────────────────────────
pivot = df.pivot_table(index='Pickup point', columns='Time Slot', values='Request id', aggfunc='count')
pivot = pivot[slot_order]  # ensure correct order

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Request Volume: Pickup Point × Time Slot', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A pivot heatmap with annotations is the best way to show three dimensions simultaneously — Pickup Point, Time Slot, and Request Count — in a compact, readable format.

##### 2. Insights?
- **Airport**: Demand peaks sharply during Evening Rush.
- **City**: Demand peaks during Morning Rush.
- Both locations have low Late Night demand.
- This confirms that Airport and City experience peak demand at different times — requiring independent scheduling strategies.

#### Chart - 16 : Status × Pickup Point × Hour (FacetGrid)

In [ ]:
# ─── Chart 16: FacetGrid - Hourly Status by Pickup Point ────────────────────
status_hour_pickup = df.groupby(['Pickup point','Hour','Status']).size().reset_index(name='Count')

g = sns.FacetGrid(status_hour_pickup, col='Pickup point', height=5, aspect=1.4)
g.map_dataframe(sns.lineplot, x='Hour', y='Count', hue='Status',
                palette={'Trip Completed':'#2ecc71','Cancelled':'#e67e22','No Cars Available':'#e74c3c'},
                marker='o')
g.add_legend()
g.set_axis_labels('Hour of Day', 'Count')
g.set_titles(col_template='{col_name} Pickup')
g.figure.suptitle('Hourly Status Breakdown by Pickup Point', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
FacetGrid line charts allow side-by-side comparison of the full hourly status pattern for each pickup point — combining three variables in one readable layout.

##### 2. Insights?
- Airport No Cars Available spikes sharply in the evening (5–9 PM).
- City Cancellations spike in early morning (5–9 AM).
- Completions at City remain more stable through the day compared to Airport.

##### 3. Business Impact?
This chart provides an operational playbook — deploy more drivers to Airport from 5 PM, and enforce anti-cancellation measures in City from 5 AM.

#### Chart - 17 : Completion Rate by Pickup Point and Time Slot (Grouped Bar)

In [ ]:
# ─── Chart 17: Completion Rate by Pickup Point and Time Slot ─────────────────
grp = df.groupby(['Pickup point','Time Slot'])
comp_rate = (grp.apply(lambda x: (x['Status'] == 'Trip Completed').sum() / len(x) * 100)
             .reset_index(name='Completion Rate (%)'))
comp_rate['Time Slot'] = pd.Categorical(comp_rate['Time Slot'], categories=slot_order, ordered=True)
comp_rate = comp_rate.sort_values('Time Slot')

fig, ax = plt.subplots(figsize=(12, 6))
comp_pivot = comp_rate.pivot(index='Time Slot', columns='Pickup point', values='Completion Rate (%)')
comp_pivot.plot(kind='bar', ax=ax, color=['#9b59b6','#1abc9c'], edgecolor='black', width=0.6)
ax.set_title('Completion Rate by Pickup Point and Time Slot', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Slot')
ax.set_ylabel('Completion Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20)
ax.legend(title='Pickup Point')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Grouped bar chart allows direct comparison of completion rates across time slots for both pickup points simultaneously — the most compact way to show three dimensions.

##### 2. Insights?
- Airport has critically low completion during Evening Rush.
- City has lowest completion during Morning Rush.
- Afternoon sees the best rates at both locations.

##### 3. Business Impact?
These two failure windows can be directly targeted with time-and-location-specific driver bonuses, making operations more precise and cost-efficient than blanket incentives.

#### Chart - 18 : Pair Plot (Numerical Features)

In [ ]:
# ─── Chart 18: Pair Plot ─────────────────────────────────────────────────────
pair_df = df_encoded[['Hour','Trip Duration (mins)','Status_encoded','Pickup_encoded']].dropna()

sns.pairplot(pair_df, diag_kind='kde', plot_kws={'alpha':0.4, 'color':'#2980b9'})
plt.suptitle('Pair Plot of Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Pair plot provides a comprehensive overview of all pairwise relationships between numerical features in a single figure — standard for multivariate EDA.

##### 2. Insights?
- Hour vs Status shows that certain hours cluster heavily in non-completed categories.
- Trip Duration shows clear separation when filtered by status — completed trips have a wide duration range while others have none.

#### Chart - 19 : Daily Request Trend Over the Data Period (Line Chart)

In [ ]:
# ─── Chart 19: Daily Requests Over Time ─────────────────────────────────────
daily = df.groupby('Date').size().reset_index(name='Requests')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily['Date'], daily['Requests'], marker='o', color='#8e44ad', linewidth=2)
ax.fill_between(daily['Date'], daily['Requests'], alpha=0.15, color='#8e44ad')
ax.set_title('Daily Ride Requests Over Dataset Period', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Requests')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A time series line chart is the most natural representation for daily volume trends — showing whether demand grew, declined, or fluctuated over the observation period.

##### 2. Insights?
Daily request volume is relatively consistent throughout the dataset period with no extreme spikes or drops — confirming the data represents steady-state operations rather than a special event period.

#### Chart - 20 : Supply vs Demand Gap by Hour (Dual Bar Chart)

In [ ]:
# ─── Chart 20: Demand vs Supply Gap ─────────────────────────────────────────
# Demand = all requests, Supply fulfilled = completed trips
demand   = df.groupby('Hour').size()
supply   = df[df['Status'] == 'Trip Completed'].groupby('Hour').size()
gap_df   = pd.DataFrame({'Demand': demand, 'Supply (Completed)': supply}).fillna(0)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(gap_df.index))
width = 0.38
ax.bar(x - width/2, gap_df['Demand'], width, label='Demand (All Requests)', color='#e74c3c', alpha=0.85)
ax.bar(x + width/2, gap_df['Supply (Completed)'], width, label='Supply (Completed)', color='#2ecc71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(gap_df.index)
ax.set_title('Demand vs Supply Gap by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Rides')
ax.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A side-by-side bar chart directly visualizes the gap between what customers want (demand) and what Uber delivers (supply) — the most direct representation of the core business problem.

##### 2. Insights?
- The gap between demand and supply is largest during **5–9 PM** (Evening Rush).
- Morning Rush (5–9 AM) also shows a substantial gap.
- Midday hours (10 AM – 4 PM) have the smallest gap — nearly balanced supply and demand.

##### 3. Business Impact?
This chart is the clearest quantification of lost business. Each red bar that towers above the green bar represents revenue left on the table. Closing even 50% of the Evening Rush gap could translate to hundreds of additional completed trips daily.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective?


Based on the EDA findings, the following data-driven recommendations are proposed:

**1. Evening Rush Airport Driver Deployment (Highest Priority)**
The biggest gap is at the Airport during 5–9 PM. Uber should introduce location-time-specific driver bonuses (e.g., ₹50 bonus per completed airport trip between 5–9 PM) to incentivize drivers to position themselves at the airport during this window.

**2. City Morning Rush Cancellation Reduction**
City cancellations during 5–9 AM are driven by drivers preferring airport routes. Introduce a cancellation penalty system during Morning Rush for City pickups, combined with a short-trip completion bonus to make City rides equally attractive.

**3. Dynamic Surge Pricing**
Implement surge pricing during Evening Rush (5–9 PM) and Morning Rush (5–9 AM) at both locations. Higher prices will naturally attract more drivers online during peak periods.

**4. Advance Booking Promotion**
Promote scheduled/advance bookings for common travel times (e.g., airport departures). Pre-assigned drivers reduce real-time supply pressure and improve completion rates.

**5. Driver Reallocation Alerts**
Build a real-time alert system that notifies drivers of surging demand at the Airport during Evening Rush, encouraging voluntary repositioning before the peak hits.

**6. Midday Driver Retention**
Since midday has best completion rates, identify and reward consistently active midday drivers to maintain this stable operational window.

# **Conclusion**

This EDA project successfully analyzed 6,745 Uber ride requests from 2016 to uncover the operational demand-supply dynamics driving ride failures.

**Key Conclusions:**

1. **More than 58% of ride requests go unfulfilled** — either cancelled by drivers or rejected due to no car availability. This is the central business problem.

2. **Evening Rush (5–9 PM) is the most critical failure window** — the highest demand combined with the lowest completion rate creates maximum revenue loss.

3. **Two distinct problems exist at two locations:**
   - Airport suffers from 'No Cars Available' especially in the evening.
   - City suffers from driver cancellations especially in the morning.

4. **Airport trips are longer and more lucrative for drivers**, which creates a systemic bias against City rides and drives up City cancellations.

5. **Midday hours (10 AM – 4 PM) are Uber's operational sweet spot** — demand is moderate, completion rates are highest, and supply-demand balance is best.

6. **Day of week has minimal effect** — operational improvements should focus on time-of-day rather than day-of-week strategies.

Implementing the suggested interventions — targeted driver incentives, surge pricing, cancellation penalties, and advance booking promotion — has the potential to significantly reduce the 58% failure rate, improving both customer satisfaction and Uber's revenue.

**Tools Used:** Python (Pandas, NumPy, Matplotlib, Seaborn) | SQL (SQLite) | Microsoft Excel

**GitHub:** https://github.com/pardeep0011/Data-Analyst-

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***